# Customer-Level Overlap Check for Training, Validation and Test Splits

## Purpose

This notebook checks whether the same customer identifier appears in more than one data partition.

The analysis uses a single file named `split_customer_ids.csv`, which should contain:

- one customer identifier column, such as `msno`;
- one split-assignment column, such as `split`, `dataset_split`, or `partition`.

Customer-level separation is important because the same customer appearing in both model-development and test data can create data leakage and overstate model performance.

## Main outputs

The notebook produces:

1. a data-quality summary;
2. a split-label summary;
3. pairwise overlap counts;
4. customers assigned to more than one split;
5. an overlap matrix;
6. a visual summary;
7. a dissertation-ready conclusion.

All evidence is saved inside a folder named `customer_overlap_outputs`.

## 1. Import libraries and set the file path

The default path below matches the folder shown on the Windows device:

`E:\customer_retention_dissertation_outputs\split_customer_ids.csv`

Change the path only if the file is stored elsewhere.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# EDIT THIS PATH ONLY IF THE FILE IS STORED ELSEWHERE
# ------------------------------------------------------------

SPLIT_ID_PATH = Path(
    r"E:\customer_retention_dissertation_outputs\split_customer_ids.csv"
)


# Leave these as None for automatic column detection.
MANUAL_ID_COLUMN = None
MANUAL_SPLIT_COLUMN = None


OUTPUT_DIR = (
    Path.cwd()
    / "customer_overlap_outputs"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("Input file:", SPLIT_ID_PATH)
print("Output folder:", OUTPUT_DIR.resolve())

## 2. Load the split-assignment file

This section:

- checks whether the file exists;
- loads the CSV as strings to preserve customer identifiers;
- displays the available columns;
- shows a short preview.

In [ ]:
if not SPLIT_ID_PATH.exists():
    raise FileNotFoundError(
        f"File not found: {SPLIT_ID_PATH}\n"
        "Check the path in Section 1 and confirm that "
        "the file extension is .csv."
    )

split_ids_raw = pd.read_csv(
    SPLIT_ID_PATH,
    dtype="string",
    low_memory=False,
)

print(
    f"Loaded {len(split_ids_raw):,} rows "
    f"and {split_ids_raw.shape[1]:,} columns."
)

print("\nAvailable columns:")
print(split_ids_raw.columns.tolist())

display(
    split_ids_raw.head(10)
)

## 3. Detect the customer-ID and split columns

The notebook searches for common customer-ID and split column names.

If automatic detection fails, return to Section 1 and set:

```python
MANUAL_ID_COLUMN = "your_customer_id_column"
MANUAL_SPLIT_COLUMN = "your_split_column"
```

In [ ]:
ID_CANDIDATES = [
    "msno",
    "customer_id",
    "customerid",
    "customer id",
    "member_id",
    "memberid",
    "member id",
    "user_id",
    "userid",
    "user id",
    "subscriber_id",
    "subscriberid",
    "subscriber id",
]

SPLIT_CANDIDATES = [
    "split",
    "dataset_split",
    "dataset split",
    "data_split",
    "data split",
    "partition",
    "set",
    "sample",
    "group",
]


def normalise_column_name(column_name):
    return (
        str(column_name)
        .strip()
        .lower()
        .replace("-", "_")
    )


column_lookup = {
    normalise_column_name(column): column
    for column in split_ids_raw.columns
}


def detect_column(
    manual_column,
    candidates,
    column_type,
):
    if manual_column is not None:
        manual_key = normalise_column_name(
            manual_column
        )

        if manual_key not in column_lookup:
            raise KeyError(
                f"The manually selected {column_type} column "
                f"'{manual_column}' was not found. "
                f"Available columns: "
                f"{split_ids_raw.columns.tolist()}"
            )

        return column_lookup[
            manual_key
        ]

    for candidate in candidates:
        candidate_key = normalise_column_name(
            candidate
        )

        if candidate_key in column_lookup:
            return column_lookup[
                candidate_key
            ]

    raise KeyError(
        f"No recognised {column_type} column was found. "
        f"Available columns: "
        f"{split_ids_raw.columns.tolist()}\n"
        f"Set the appropriate MANUAL_*_COLUMN value "
        f"in Section 1."
    )


id_column = detect_column(
    manual_column=MANUAL_ID_COLUMN,
    candidates=ID_CANDIDATES,
    column_type="customer-ID",
)

split_column = detect_column(
    manual_column=MANUAL_SPLIT_COLUMN,
    candidates=SPLIT_CANDIDATES,
    column_type="split",
)

print("Detected customer-ID column:", id_column)
print("Detected split column:", split_column)

## 4. Clean customer identifiers and standardise split labels

The notebook standardises common split labels:

- `train` and `training` → `training`
- `val`, `valid`, and `validation` → `validation`
- `test` and `testing` → `test`

Missing customer identifiers and unrecognised split labels are reported separately.

In [ ]:
split_ids = split_ids_raw[
    [
        id_column,
        split_column,
    ]
].copy()

split_ids[id_column] = (
    split_ids[id_column]
    .astype("string")
    .str.strip()
)

split_ids[split_column] = (
    split_ids[split_column]
    .astype("string")
    .str.strip()
    .str.lower()
)

invalid_id_mask = (
    split_ids[id_column].isna()
    | split_ids[id_column].eq("")
    | split_ids[id_column].str.lower().isin(
        [
            "nan",
            "none",
            "null",
            "<na>",
        ]
    )
)

split_label_mapping = {
    "train": "training",
    "training": "training",
    "tr": "training",
    "val": "validation",
    "valid": "validation",
    "validation": "validation",
    "dev": "validation",
    "test": "test",
    "testing": "test",
    "te": "test",
}

split_ids["standard_split"] = (
    split_ids[split_column]
    .map(split_label_mapping)
)

invalid_split_mask = (
    split_ids["standard_split"].isna()
)

valid_split_ids = split_ids.loc[
    ~invalid_id_mask
    & ~invalid_split_mask
].copy()

unknown_split_values = (
    split_ids.loc[
        invalid_split_mask,
        split_column,
    ]
    .dropna()
    .unique()
    .tolist()
)

print(
    "Rows with missing or invalid customer IDs:",
    f"{int(invalid_id_mask.sum()):,}",
)

print(
    "Rows with unrecognised split labels:",
    f"{int(invalid_split_mask.sum()):,}",
)

if unknown_split_values:
    print(
        "Unrecognised split values:",
        unknown_split_values,
    )

print(
    "Valid rows retained for overlap analysis:",
    f"{len(valid_split_ids):,}",
)

## 5. Data-quality and split-distribution summary

This section reports:

- total rows;
- valid and invalid identifiers;
- valid and invalid split labels;
- unique customers;
- duplicate customer–split rows;
- number of customers in each split.

In [ ]:
duplicate_customer_split_rows = int(
    valid_split_ids.duplicated(
        subset=[
            id_column,
            "standard_split",
        ],
        keep=False,
    ).sum()
)

repeated_customer_split_rows = int(
    valid_split_ids.duplicated(
        subset=[
            id_column,
            "standard_split",
        ],
        keep="first",
    ).sum()
)

quality_summary = pd.DataFrame(
    [
        {
            "total_rows": int(
                len(split_ids)
            ),
            "valid_rows_for_analysis": int(
                len(valid_split_ids)
            ),
            "missing_or_invalid_customer_ids": int(
                invalid_id_mask.sum()
            ),
            "unrecognised_split_labels": int(
                invalid_split_mask.sum()
            ),
            "unique_customers": int(
                valid_split_ids[
                    id_column
                ].nunique()
            ),
            "rows_in_duplicate_customer_split_groups": (
                duplicate_customer_split_rows
            ),
            "duplicate_customer_split_rows_after_first": (
                repeated_customer_split_rows
            ),
        }
    ]
)

split_distribution = (
    valid_split_ids
    .groupby(
        "standard_split",
        dropna=False,
    )
    .agg(
        rows=(
            id_column,
            "size",
        ),
        unique_customers=(
            id_column,
            "nunique",
        ),
    )
    .reset_index()
    .sort_values(
        "standard_split"
    )
)

quality_summary.to_csv(
    OUTPUT_DIR
    / "01_customer_id_quality_summary.csv",
    index=False,
)

split_distribution.to_csv(
    OUTPUT_DIR
    / "02_split_distribution.csv",
    index=False,
)

display(
    quality_summary
)

display(
    split_distribution
)

## 6. Identify customers assigned to more than one split

A customer should belong to only one of the three partitions.

This section identifies any customer whose identifier appears under multiple split labels.

In [ ]:
customer_split_counts = (
    valid_split_ids
    .groupby(id_column)[
        "standard_split"
    ]
    .nunique()
)

overlapping_customer_index = (
    customer_split_counts.loc[
        customer_split_counts > 1
    ]
    .index
)

multi_split_customer_details = (
    valid_split_ids.loc[
        valid_split_ids[
            id_column
        ].isin(
            overlapping_customer_index
        )
    ]
    .drop_duplicates(
        subset=[
            id_column,
            "standard_split",
        ]
    )
    .sort_values(
        [
            id_column,
            "standard_split",
        ]
    )
    .reset_index(
        drop=True
    )
)

multi_split_customer_summary = (
    multi_split_customer_details
    .groupby(id_column)[
        "standard_split"
    ]
    .agg(
        lambda values: ", ".join(
            sorted(
                set(values)
            )
        )
    )
    .reset_index(
        name="assigned_splits"
    )
)

multi_split_customer_summary[
    "number_of_splits"
] = (
    multi_split_customer_summary[
        "assigned_splits"
    ]
    .str.count(",")
    + 1
)

multi_split_customer_summary.to_csv(
    OUTPUT_DIR
    / "03_customers_in_multiple_splits.csv",
    index=False,
)

print(
    "Customers assigned to more than one split:",
    f"{len(multi_split_customer_summary):,}",
)

display(
    multi_split_customer_summary.head(20)
)

## 7. Calculate pairwise and three-way overlap

The pairwise comparisons are:

- training versus validation;
- training versus test;
- validation versus test.

The notebook also checks whether any customer appears in all three partitions.

In [ ]:
expected_splits = [
    "training",
    "validation",
    "test",
]

customer_sets = {
    split_name: set(
        valid_split_ids.loc[
            valid_split_ids[
                "standard_split"
            ]
            == split_name,
            id_column,
        ]
        .dropna()
        .tolist()
    )
    for split_name in expected_splits
}

pairwise_comparisons = [
    ("training", "validation"),
    ("training", "test"),
    ("validation", "test"),
]

overlap_summary_rows = []
overlap_detail_frames = []

for first_split, second_split in (
    pairwise_comparisons
):
    first_customers = customer_sets[
        first_split
    ]

    second_customers = customer_sets[
        second_split
    ]

    overlap = (
        first_customers
        & second_customers
    )

    comparison_name = (
        f"{first_split}_vs_"
        f"{second_split}"
    )

    overlap_summary_rows.append(
        {
            "comparison": comparison_name,
            "first_split": first_split,
            "second_split": second_split,
            "first_unique_customers": int(
                len(first_customers)
            ),
            "second_unique_customers": int(
                len(second_customers)
            ),
            "overlapping_customers": int(
                len(overlap)
            ),
            "overlap_percent_of_first": (
                100
                * len(overlap)
                / len(first_customers)
                if first_customers
                else np.nan
            ),
            "overlap_percent_of_second": (
                100
                * len(overlap)
                / len(second_customers)
                if second_customers
                else np.nan
            ),
        }
    )

    if overlap:
        overlap_detail_frames.append(
            pd.DataFrame(
                {
                    "comparison": comparison_name,
                    "customer_id": sorted(
                        overlap
                    ),
                }
            )
        )

three_way_overlap = (
    customer_sets["training"]
    & customer_sets["validation"]
    & customer_sets["test"]
)

overlap_summary_rows.append(
    {
        "comparison": (
            "training_vs_validation_vs_test"
        ),
        "first_split": "all_three",
        "second_split": "all_three",
        "first_unique_customers": np.nan,
        "second_unique_customers": np.nan,
        "overlapping_customers": int(
            len(three_way_overlap)
        ),
        "overlap_percent_of_first": np.nan,
        "overlap_percent_of_second": np.nan,
    }
)

if three_way_overlap:
    overlap_detail_frames.append(
        pd.DataFrame(
            {
                "comparison": (
                    "training_vs_validation_vs_test"
                ),
                "customer_id": sorted(
                    three_way_overlap
                ),
            }
        )
    )

overlap_summary = pd.DataFrame(
    overlap_summary_rows
)

if overlap_detail_frames:
    overlap_details = pd.concat(
        overlap_detail_frames,
        ignore_index=True,
    )
else:
    overlap_details = pd.DataFrame(
        columns=[
            "comparison",
            "customer_id",
        ]
    )

overlap_summary.to_csv(
    OUTPUT_DIR
    / "04_customer_overlap_summary.csv",
    index=False,
)

overlap_details.to_csv(
    OUTPUT_DIR
    / "05_overlapping_customer_ids.csv",
    index=False,
)

display(
    overlap_summary.style.format(
        {
            "overlap_percent_of_first": "{:.6f}",
            "overlap_percent_of_second": "{:.6f}",
        }
    )
)

## 8. Create the overlap matrix and visual summary

The diagonal contains the number of unique customers in each split.

The off-diagonal values contain the number of customers shared between two partitions.

In [ ]:
overlap_matrix = pd.DataFrame(
    index=expected_splits,
    columns=expected_splits,
    dtype=int,
)

for row_split in expected_splits:
    for column_split in expected_splits:
        if row_split == column_split:
            value = len(
                customer_sets[
                    row_split
                ]
            )
        else:
            value = len(
                customer_sets[
                    row_split
                ]
                & customer_sets[
                    column_split
                ]
            )

        overlap_matrix.loc[
            row_split,
            column_split,
        ] = int(value)

overlap_matrix = (
    overlap_matrix.astype(int)
)

overlap_matrix.to_csv(
    OUTPUT_DIR
    / "06_customer_overlap_matrix.csv"
)

display(
    overlap_matrix
)


pairwise_plot_data = (
    overlap_summary.loc[
        overlap_summary[
            "comparison"
        ]
        != "training_vs_validation_vs_test",
        [
            "comparison",
            "overlapping_customers",
        ],
    ]
    .copy()
)

figure, axis = plt.subplots(
    figsize=(9, 5)
)

bars = axis.bar(
    pairwise_plot_data[
        "comparison"
    ],
    pairwise_plot_data[
        "overlapping_customers"
    ],
)

axis.set_title(
    "Customer-ID overlap between data splits"
)

axis.set_xlabel(
    "Split comparison"
)

axis.set_ylabel(
    "Overlapping unique customers"
)

axis.tick_params(
    axis="x",
    rotation=20,
)

axis.grid(
    axis="y",
    alpha=0.3,
)

for bar in bars:
    height = bar.get_height()

    axis.text(
        bar.get_x()
        + bar.get_width() / 2,
        height,
        f"{int(height):,}",
        ha="center",
        va="bottom",
    )

figure.tight_layout()

figure.savefig(
    OUTPUT_DIR
    / "07_customer_overlap_counts.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

## 9. Final leakage conclusion

The result passes only when:

- training and validation share zero customers;
- training and test share zero customers;
- validation and test share zero customers;
- no customer appears in all three splits.

In [ ]:
pairwise_overlap_total = int(
    overlap_summary.loc[
        overlap_summary[
            "comparison"
        ]
        != "training_vs_validation_vs_test",
        "overlapping_customers",
    ].sum()
)

three_way_overlap_count = int(
    len(three_way_overlap)
)

multi_split_customer_count = int(
    len(
        multi_split_customer_summary
    )
)

missing_id_count = int(
    invalid_id_mask.sum()
)

invalid_split_count = int(
    invalid_split_mask.sum()
)


if (
    pairwise_overlap_total == 0
    and three_way_overlap_count == 0
    and multi_split_customer_count == 0
):
    status = "PASSED"

    conclusion = (
        "PASSED: No customer identifiers overlap "
        "between training, validation and test."
    )

    dissertation_statement = (
        "Customer-level independence was verified "
        "using the retained anonymised customer "
        "identifiers. No customer identifiers were "
        "shared between the training, validation and "
        "test partitions, reducing the risk of "
        "customer-level data leakage."
    )

else:
    status = "WARNING"

    conclusion = (
        "WARNING: Customer-level overlap was detected "
        "between one or more data partitions."
    )

    dissertation_statement = (
        "Customer-level overlap was detected between "
        "one or more data partitions. The data should "
        "be repartitioned using unique customer "
        "identifiers before the final model "
        "performance is treated as independent."
    )


conclusion_lines = [
    f"Status: {status}",
    conclusion,
    "",
    (
        "Pairwise overlap total: "
        f"{pairwise_overlap_total:,}"
    ),
    (
        "Customers in all three splits: "
        f"{three_way_overlap_count:,}"
    ),
    (
        "Customers assigned to multiple splits: "
        f"{multi_split_customer_count:,}"
    ),
    (
        "Missing or invalid customer IDs: "
        f"{missing_id_count:,}"
    ),
    (
        "Rows with unrecognised split labels: "
        f"{invalid_split_count:,}"
    ),
    "",
    "Suggested dissertation statement:",
    dissertation_statement,
]

conclusion_text = "\n".join(
    conclusion_lines
)

print(conclusion_text)

conclusion_path = (
    OUTPUT_DIR
    / "08_customer_overlap_conclusion.txt"
)

conclusion_path.write_text(
    conclusion_text,
    encoding="utf-8",
)

final_status = pd.DataFrame(
    [
        {
            "status": status,
            "pairwise_overlap_total": (
                pairwise_overlap_total
            ),
            "three_way_overlap_count": (
                three_way_overlap_count
            ),
            "multi_split_customer_count": (
                multi_split_customer_count
            ),
            "missing_or_invalid_customer_ids": (
                missing_id_count
            ),
            "unrecognised_split_labels": (
                invalid_split_count
            ),
            "dissertation_statement": (
                dissertation_statement
            ),
        }
    ]
)

final_status.to_csv(
    OUTPUT_DIR
    / "09_final_customer_overlap_status.csv",
    index=False,
)

## 10. Output checklist

After the notebook finishes, confirm that the output folder contains:

- `01_customer_id_quality_summary.csv`
- `02_split_distribution.csv`
- `03_customers_in_multiple_splits.csv`
- `04_customer_overlap_summary.csv`
- `05_overlapping_customer_ids.csv`
- `06_customer_overlap_matrix.csv`
- `07_customer_overlap_counts.png`
- `08_customer_overlap_conclusion.txt`
- `09_final_customer_overlap_status.csv`

### Interpretation

**PASSED** means the training, validation and test partitions are independent at customer level.

**WARNING** means that at least one customer identifier appears in more than one partition. In that case, rebuild the splits by unique customer before treating test performance as independent.

In [ ]:
expected_output_files = [
    "01_customer_id_quality_summary.csv",
    "02_split_distribution.csv",
    "03_customers_in_multiple_splits.csv",
    "04_customer_overlap_summary.csv",
    "05_overlapping_customer_ids.csv",
    "06_customer_overlap_matrix.csv",
    "07_customer_overlap_counts.png",
    "08_customer_overlap_conclusion.txt",
    "09_final_customer_overlap_status.csv",
]

output_check = pd.DataFrame(
    {
        "file": expected_output_files,
        "exists": [
            (
                OUTPUT_DIR
                / file_name
            ).exists()
            for file_name in (
                expected_output_files
            )
        ],
    }
)

display(
    output_check
)

if output_check[
    "exists"
].all():
    print(
        "\nNOTEBOOK COMPLETED SUCCESSFULLY"
    )
    print(
        "Output folder:",
        OUTPUT_DIR.resolve(),
    )
else:
    missing_files = output_check.loc[
        ~output_check["exists"],
        "file",
    ].tolist()

    raise FileNotFoundError(
        "Some expected outputs were not created: "
        + ", ".join(missing_files)
    )